# Versión 2 · Descarga de datos
## Nubosidad, precipitación y desempeño de las plantas solares fotovoltaicas de Colombia

**Idea:** el factor de planta (energía real ÷ energía máxima posible) de las plantas solares de Colombia cambia **entre regiones** (Caribe, Magdalena Medio, Llanos, valles interandinos) y **entre meses** (temporadas secas y lluviosas). La hipótesis es que esas diferencias se explican sobre todo por la **nubosidad** y la **precipitación**. El objetivo es medir cuánto, y si esa relación cambia en el espacio.

**Unidad de análisis:** cada planta solar que reporta a XM, mes a mes, entre `INICIO` y `FIN` (por defecto 2019–2025). El resultado de este notebook es una **tabla panel planta × mes** lista para correlaciones, regresiones y estadística espacial.

| Tipo | Dato | Fuente | Resolución | Cuenta |
|---|---|---|---|---|
| Punto | Generación real por planta solar → factor de planta mensual | XM (API) | horaria → mensual | No |
| Punto | Despacho ideal, generación programada (despacho y redespacho), disponibilidad, irradiación y temperatura medidas en cada planta | XM (API) | horaria / diaria → mensual | No |
| Punto | Ubicación de las plantas | Tabla base (Global Energy Monitor) + OpenStreetMap (automático) + revisión manual | — | No |
| Punto | Nubosidad, radiación, índice de claridad y lluvia en cada planta | NASA POWER (API) | mensual, ~0,5° | No |
| Punto | **Estaciones del IDEAM:** precipitación, temperatura y humedad observadas | IDEAM, Datos Abiertos (datos.gov.co) — **notebook aparte** `descarga_estaciones_ideam.ipynb` | 10 min → diaria → mensual | No |
| Punto | **Normales climatológicas IDEAM 1981–2010:** lluvia, días de lluvia, temperatura, **brillo solar**, humedad, evaporación | IDEAM (Excel «Promedios Climatológicos 1981–2010») | mensual (climatología) | No |
| Punto | **Precipitación diaria CHIRPS en cada planta** → lluvia y días de lluvia por mes | UCSB/CHC vía ERDDAP de NOAA | diaria, 0,05° | No |
| Área | Departamentos y municipios | geoBoundaries | — | No |
| Área | Régimen de lluvias (unimodal/bimodal) | IDEAM, OGC `Clima_Analisis_Climatico` (32) | — | No |
| Área | Zonas de radiación solar promedio multianual | IDEAM, OGC `Clima_Potencial_Energetico` (25) | — | No |
| Superficie | Precipitación mensual CHIRPS v2.0 | UCSB/CHC vía ERDDAP de NOAA (respaldo: GeoTIFF de UCSB) | 0,05° (~5 km) | No |
| Superficie | Frecuencia de nubes MODIS (climatología mensual) | EarthEnv (Wilson y Jetz, 2016) | ~1 km | No |
| Superficie | Elevación en cada planta | Copernicus GLO-30 (AWS) | 30 m | No |

**No necesitas ninguna cuenta.** Con el periodo 2019–2025 la primera ejecución puede tardar **alrededor de 1 hora** (sobre todo XM); las siguientes reutilizan lo descargado.

**Sección 8:** desviaciones entre la generación real y la programada por XM (horarias, diarias y mensuales), por planta y por región, con mapas.

## 0. Preparación

In [ ]:
import os, sys, subprocess
from pathlib import Path
PAQUETES = ["numpy", "pandas", "requests", "xarray", "netCDF4", "geopandas", "rasterio", "shapely",
            "pydataxm", "matplotlib"]
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/solar_colombia_v2"); CARPETA.mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES], check=True)
print("Carpeta de trabajo:", os.getcwd())

## 1. Parámetros

In [ ]:
from datetime import date

INICIO = date(2019, 1, 1)        # periodo de análisis (años completos; la primera planta solar grande, El Paso, entró en 2019)
FIN = date(2025, 12, 31)         # último año completo. Puedes ampliarlo: lo ya descargado no se vuelve a pedir
PERIODO = f"{INICIO:%Y%m}-{FIN:%Y%m}"   # va en el nombre de los archivos que dependen del periodo
BBOX_COL = (-79.1, -4.3, -66.8, 13.5)          # lon_min, lat_min, lon_max, lat_max (Colombia continental)

# Tabla base de ubicaciones (Global Energy Monitor). 'clave' = palabra para encontrar la planta en XM.
# Coordenadas 'aprox' → verifícalas en Google Earth. Agrega filas si una planta no se ubica sola.
UBICACIONES_BASE = [
    dict(clave="FUNDACION",      nombre="Fundación (Enel)",            lat=10.4812, lon=-74.2263, dpto="Magdalena", precision="exacta GEM"),
    dict(clave="LA LOMA",        nombre="La Loma (Enel)",              lat=9.7063,  lon=-73.5222, dpto="Cesar",     precision="aprox GEM"),
    dict(clave="TEPUY",          nombre="Tepuy (EPM)",                 lat=5.4689,  lon=-74.6711, dpto="Caldas",    precision="aprox GEM"),
    dict(clave="BOSQUES SOLARES",nombre="Bosques Solares de los Llanos",lat=4.2180, lon=-72.0221, dpto="Meta",      precision="exacta GEM"),
    dict(clave="SINCE",          nombre="Sincé",                       lat=9.2473,  lon=-75.1488, dpto="Sucre",     precision="aprox GEM"),
    dict(clave="CAMPANO",        nombre="Campano",                     lat=9.1064,  lon=-75.3989, dpto="Córdoba",   precision="aprox GEM"),
    dict(clave="PLANETA RICA",   nombre="Planeta Rica",                lat=8.4416,  lon=-75.6025, dpto="Córdoba",   precision="exacta GEM"),
]

DESCARGAR = {"areas": True, "xm": True, "ubicaciones_osm": True, "power": True,
             "chirps": True, "nubes_earthenv": True, "elevacion": True,
             "chirps_diario": True, "xm_despacho": True}

UMBRAL_LLUVIA_MM = 1.0     # un día cuenta como lluvioso si llueve al menos esto

BASE = Path.cwd()
RAW, PROC, OUT = BASE / "datos" / "crudos", BASE / "datos" / "procesados", BASE / "resultados"
for p in (RAW, PROC, OUT):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
import os, json, re, time, unicodedata, difflib, urllib.request
import certifi
# GDAL (rasterio /vsicurl) en Windows a veces no encuentra los certificados SSL y no puede abrir archivos remotos
os.environ.setdefault("CURL_CA_BUNDLE", certifi.where())
os.environ.setdefault("GDAL_HTTP_TIMEOUT", "180")
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

def get(url, intentos=4, **kw):
    espera = kw.pop("timeout", 300)
    for i in range(intentos):
        try:
            r = requests.get(url, timeout=espera, **kw); r.raise_for_status(); return r
        except Exception as e:
            ultimo = e
            print(f"   reintento {i + 1}: {str(e)[:200]}"); time.sleep(10)
    raise RuntimeError(f"{url[:150]} → {ultimo}")

def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().upper()
    t = re.sub(r"\b(PARQUE|PLANTA|GRANJA|SOLAR|FOTOVOLTAIC[OA]|PSF|PV|SAS|S\.A\.S|ESP|DE|DEL|LA|EL|LOS)\b", " ", t)
    return re.sub(r"[^A-Z0-9 ]", " ", re.sub(r"\s+", " ", t)).strip()

def meses(a, b):
    return pd.period_range(pd.Timestamp(a), pd.Timestamp(b), freq="M")

## 2. Datos tipo ÁREA
### 2.1 Departamentos y municipios de Colombia (geoBoundaries)

In [ ]:
def geoboundaries(nivel):
    with urllib.request.urlopen(f"https://www.geoboundaries.org/api/current/gbOpen/COL/{nivel}/", timeout=120) as r:
        return gpd.read_file(json.load(r)["gjDownloadURL"]).to_crs(4326)

for nivel, nombre in (("ADM1", "departamentos"), ("ADM2", "municipios")):
    f = PROC / f"{nombre}.gpkg"
    if DESCARGAR["areas"] and not f.exists():
        geoboundaries(nivel).to_file(f, driver="GPKG")
dptos = gpd.read_file(PROC / "departamentos.gpkg")
mpios = gpd.read_file(PROC / "municipios.gpkg")
print(f"{len(dptos)} departamentos · {len(mpios)} municipios")

### 2.2 Capas climáticas del IDEAM (servicio OGC API Features)
- **Régimen pluviométrico 1981–2010** (campo `regimen`): agrupa las plantas en regiones con **una** o **dos** temporadas de lluvia. Sirve para comparar la estacionalidad del factor de planta.
- **Radiación solar global promedio multianual** (campo `rango`, kWh/m²·día): el recurso solar esperado de cada zona.

In [ ]:
def ogc_ideam(servicio, coleccion, bbox, lote=1000):
    url = (f"https://visualizador.ideam.gov.co/gisserver/rest/services/{servicio}"
           f"/OGCFeatureServer/collections/{coleccion}/items")
    feats, offset = [], 0
    while True:
        fc = get(url, params={"f": "json", "bbox": ",".join(map(str, bbox)), "limit": lote, "offset": offset}).json()
        nuevos = fc.get("features", []); feats += nuevos
        if len(nuevos) < lote or not any(l.get("rel") == "next" for l in fc.get("links", [])):
            break
        offset += lote
    g = gpd.GeoDataFrame.from_features(feats, crs=4326)
    g["geometry"] = g.geometry.buffer(0)
    return g

capas_ideam = {"regimen_pluviometrico": ("Clima_Analisis_Climatico", 32),
               "zonas_radiacion": ("Clima_Potencial_Energetico", 25)}
for nombre, (serv, col) in capas_ideam.items():
    f = PROC / f"{nombre}_ideam.gpkg"
    if DESCARGAR["areas"] and not f.exists():
        print("Descargando", nombre, "…")
        ogc_ideam(serv, col, BBOX_COL).to_file(f, driver="GPKG")
regimen = gpd.read_file(PROC / "regimen_pluviometrico_ideam.gpkg")
zrad = gpd.read_file(PROC / "zonas_radiacion_ideam.gpkg")
lim = zrad["rango"].str.extract(r"([\d.]+)\s*-\s*([\d.]+)").astype(float)
zrad["rad_kwh_m2_dia"] = lim.mean(axis=1)
print("Regímenes:", regimen["regimen"].value_counts().to_dict())
print("Rangos de radiación:", sorted(zrad["rango"].dropna().unique()))

## 3. Datos tipo PUNTO
### 3.1 Plantas solares en XM y su generación mensual
1. Listado de recursos de XM → se guardan los **solares**.
2. Generación horaria `Gene/Recurso`, mes a mes → **energía mensual (MWh)**.
3. Capacidad: se intenta la métrica de XM `CapEfecNeta`. Si no está disponible, se usa como respaldo la **máxima generación horaria observada** (`cap_proxy_mw`).
4. **Factor de planta mensual** = energía ÷ (capacidad × horas del mes).

In [ ]:
from pydataxm.pydataxm import ReadDB
api = ReadDB()

def col(df, *claves):
    for c in df.columns:
        if any(k in c.lower() for k in claves):
            return c
    raise KeyError(f"{claves} no está en {list(df.columns)}")

f_rec = PROC / f"xm_recursos_solares_{FIN:%Y%m}.csv"   # plantas registradas al final del periodo
if DESCARGAR["xm"] and not f_rec.exists():
    rec = api.request_data("ListadoRecursos", "Sistema", FIN, FIN)
    rec.to_csv(RAW / f"xm_listado_recursos_{FIN:%Y%m}.csv", index=False)
    rec[rec.astype(str).apply(lambda f: " ".join(f).upper(), axis=1).str.contains("SOLAR")].to_csv(f_rec, index=False)
rec_sol = pd.read_csv(f_rec, dtype=str)
C_COD, C_NOM = col(rec_sol, "code", "codigo"), col(rec_sol, "name", "nombre")
CODIGOS = set(rec_sol[C_COD])
print(f"{len(rec_sol)} recursos solares en XM")
rec_sol[[C_COD, C_NOM]].head(40)

In [ ]:
# Consulta DIRECTA a la API de XM (sin pydataxm): evita el error de asyncio en Jupyter / Python ≥ 3.12
XM_URL = "https://servapibi.xm.com.co/{}"

def xm_api(metrica, entidad, inicio, fin, endpoint="hourly", tramo=30):
    filas, a, fin = [], pd.Timestamp(inicio), pd.Timestamp(fin)
    while a <= fin:
        b = min(a + pd.Timedelta(days=tramo - 1), fin)
        cuerpo = {"MetricId": metrica, "StartDate": f"{a:%Y-%m-%d}", "EndDate": f"{b:%Y-%m-%d}",
                  "Entity": entidad, "Filter": []}
        for intento in range(4):
            try:
                r = requests.post(XM_URL.format(endpoint), json=cuerpo, timeout=300)
                r.raise_for_status()
                break
            except Exception as e:
                print(f"   reintento {intento + 1}: {e}"); time.sleep(10)
        else:
            raise RuntimeError(f"XM no respondió: {metrica} {a:%Y-%m-%d}")
        for item in r.json().get("Items", []):
            for clave, entidades in item.items():
                if clave.endswith("Entities"):                 # HourlyEntities, DailyEntities…
                    for ent in entidades:
                        filas.append({"Date": item.get("Date"), **ent.get("Values", {})})
        a = b + pd.Timedelta(days=1)
    return pd.DataFrame(filas)

f_gen = PROC / f"xm_generacion_mensual_{PERIODO}.csv"
if DESCARGAR["xm"] and not f_gen.exists():
    partes = []
    for m in meses(INICIO, FIN):
        f_m = RAW / "xm" / f"gene_{m}.csv"; f_m.parent.mkdir(parents=True, exist_ok=True)
        if not f_m.exists():
            print("XM", m, "…")
            g = xm_api("Gene", "Recurso", m.start_time.date(), m.end_time.date())
            if g.empty:
                print("   sin datos"); pd.DataFrame(columns=["Date", "code"]).to_csv(f_m, index=False)
            else:
                g[g["code"].astype(str).isin(CODIGOS)].to_csv(f_m, index=False)
        partes.append(pd.read_csv(f_m, dtype={"code": str}))
    g = pd.concat(partes, ignore_index=True)
    horas = [c for c in g.columns if c.lower().startswith("hour")]
    g[horas] = g[horas].apply(pd.to_numeric, errors="coerce") / 1000           # kWh → MWh
    g["mes"] = pd.to_datetime(g["Date"]).dt.to_period("M")
    g["mwh_dia"] = g[horas].sum(axis=1, min_count=1)
    g["max_hora_mwh"] = g[horas].max(axis=1)
    mensual = g.groupby(["code", "mes"]).agg(energia_mwh=("mwh_dia", "sum"), dias_con_dato=("mwh_dia", "count"),
                                             max_hora_mwh=("max_hora_mwh", "max")).reset_index()
    mensual.rename(columns={"code": "codigo_xm"}).to_csv(f_gen, index=False)
mensual = pd.read_csv(f_gen, dtype={"codigo_xm": str})
mensual["mes"] = pd.PeriodIndex(mensual["mes"], freq="M")

# Capacidad: métrica diaria de XM (CapEfecNeta) si existe; si no, proxy = máxima generación horaria observada
cap = mensual.groupby("codigo_xm")["max_hora_mwh"].max().rename("cap_proxy_mw").to_frame()
try:
    ce = xm_api("CapEfecNeta", "Recurso", pd.Timestamp(FIN) - pd.Timedelta(days=6), FIN, endpoint="daily")
    ce = ce[ce["code"].astype(str).isin(CODIGOS)]
    val = next(c for c in ce.columns if c.lower() == "value")
    cruda = ce.assign(v=pd.to_numeric(ce[val], errors="coerce")).groupby("code")["v"].last()
    cap = cap.join(cruda.rename("cap_xm_mw"))
    if (cap["cap_xm_mw"] / cap["cap_proxy_mw"]).median() > 100:   # venía en kW → MW
        cap["cap_xm_mw"] /= 1000
except Exception as e:
    print("CapEfecNeta no disponible:", e)
if "cap_xm_mw" in cap:
    razon = cap["cap_xm_mw"] / cap["cap_proxy_mw"]
    dudosa = ~razon.between(0.8, 2.5)          # valores inconsistentes → se usa el proxy
    print(f"Capacidad XM usada en {(~dudosa).sum()} plantas; proxy en {dudosa.sum()} (revisa xm_capacidad.csv)")
    cap["capacidad_mw"] = cap["cap_xm_mw"].where(~dudosa, cap["cap_proxy_mw"])
else:
    cap["capacidad_mw"] = cap["cap_proxy_mw"]
cap.to_csv(PROC / "xm_capacidad.csv")
mensual = mensual.merge(cap[["capacidad_mw"]], left_on="codigo_xm", right_index=True)
mensual["horas_mes"] = mensual["mes"].dt.days_in_month * 24
mensual["factor_planta"] = mensual["energia_mwh"] / (mensual["capacidad_mw"] * mensual["horas_mes"])
mensual.loc[mensual["dias_con_dato"] < mensual["mes"].dt.days_in_month - 2, "factor_planta"] = np.nan   # meses incompletos
print(f"{mensual.codigo_xm.nunique()} plantas con generación · {len(mensual)} planta-mes")
mensual.groupby("codigo_xm")["factor_planta"].describe().round(3)

### 3.1b Despacho, generación programada y medidas de las plantas solares (XM)
Además de la generación real (`Gene`), XM publica para cada recurso:

| Métrica XM | Qué es | Paso | Se guarda como |
|---|---|---|---|
| `GeneIdea` | Despacho ideal: lo que habría generado sin restricciones de red | horario | `gen_ideal_mwh` (suma mensual) |
| `GeneProgDesp` | Generación programada en el despacho del día anterior | horario | `gen_prog_despacho_mwh` |
| `GeneProgRedesp` | Generación programada en el redespacho (ajustes durante el día) | horario | `gen_prog_redespacho_mwh` |
| `DispoReal` | Disponibilidad real de la planta | horario | `dispo_real_mw` (media mensual) |
| `DispoDeclarada` | Potencia máxima declarada en la oferta | horario | `dispo_declarada_mw` (media mensual) |
| `IrrGlobal` | Irradiación global horizontal medida en la planta | diario | `xm_irr_global` (media diaria del mes) |
| `IrrPanel` | Irradiación en el plano del panel | diario | `xm_irr_panel` |
| `TempPanel` | Temperatura del panel | diario | `xm_temp_panel_c` |
| `TempAmbSolar` | Temperatura ambiente en la planta | diario | `xm_temp_amb_c` |

Se calculan además:
- `desvio_despacho_mwh` = generación real − programada en el despacho (negativo: produjo menos de lo previsto),
- `cumplimiento_despacho` = real ÷ programada.

**Para qué sirven:** la **diferencia entre lo programado y lo real** es justamente la parte de la generación que no se pudo prever, en buena medida por **nubes y lluvia** del día. La disponibilidad permite separar meses con fallas o mantenimientos (baja generación que no es por el clima). La irradiación y la temperatura medidas en la propia planta son la mejor referencia para validar NASA POWER.

Las plantas pequeñas que no son despachadas centralmente no tienen despacho programado: sus columnas quedan vacías. Las unidades de irradiación son las que reporta XM: revísalas en `ListadoMetricas`.

In [ ]:
METRICAS_XM_HORARIAS = {   # métrica: (columna, agregación mensual); los valores llegan en kWh o kW → se pasan a MWh / MW
    "GeneIdea": ("gen_ideal_mwh", "sum"),
    "GeneProgDesp": ("gen_prog_despacho_mwh", "sum"),
    "GeneProgRedesp": ("gen_prog_redespacho_mwh", "sum"),
    "DispoReal": ("dispo_real_mw", "mean"),
    "DispoDeclarada": ("dispo_declarada_mw", "mean"),
}
METRICAS_XM_DIARIAS = {
    "IrrGlobal": "xm_irr_global", "IrrPanel": "xm_irr_panel",
    "TempPanel": "xm_temp_panel_c", "TempAmbSolar": "xm_temp_amb_c",
}

def xm_metrica_mensual(metrica, endpoint):
    """Descarga mes a mes (guardando cada mes en datos/crudos/xm) y deja solo las plantas solares."""
    partes = []
    for m in meses(INICIO, FIN):
        f_m = RAW / "xm" / f"{metrica}_{m}.csv"; f_m.parent.mkdir(parents=True, exist_ok=True)
        if DESCARGAR["xm_despacho"] and not f_m.exists():
            try:
                d = xm_api(metrica, "Recurso", m.start_time.date(), m.end_time.date(), endpoint=endpoint)
            except Exception as e:                       # no se guarda: se reintenta en la próxima ejecución
                print(f"   {metrica} {m}: {e}"); continue
            (d[d["code"].astype(str).isin(CODIGOS)] if "code" in d else pd.DataFrame(columns=["Date", "code"])).to_csv(f_m, index=False)
        if f_m.exists():
            partes.append(pd.read_csv(f_m, dtype={"code": str}))
    d = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    if d.empty or "code" not in d:
        return pd.DataFrame(columns=["codigo_xm", "mes"])
    d["mes"] = pd.to_datetime(d["Date"]).dt.to_period("M").astype(str)
    return d.rename(columns={"code": "codigo_xm"})

xm_extra = None
for metrica, (columna, agg) in METRICAS_XM_HORARIAS.items():
    print("XM", metrica, "…")
    d = xm_metrica_mensual(metrica, "hourly")
    if d.empty:
        continue
    horas = [c for c in d.columns if c.lower().startswith("hour")]
    d[horas] = d[horas].apply(pd.to_numeric, errors="coerce") / 1000                 # kWh → MWh · kW → MW
    d["dia"] = d[horas].sum(axis=1, min_count=1) if agg == "sum" else d[horas].mean(axis=1)
    m = d.groupby(["codigo_xm", "mes"])["dia"].agg(agg).rename(columna).reset_index()
    xm_extra = m if xm_extra is None else xm_extra.merge(m, on=["codigo_xm", "mes"], how="outer")
for metrica, columna in METRICAS_XM_DIARIAS.items():
    print("XM", metrica, "…")
    d = xm_metrica_mensual(metrica, "daily")
    val = next((c for c in d.columns if c.lower() == "value"), None)
    if d.empty or val is None:
        continue
    m = d.assign(v=pd.to_numeric(d[val], errors="coerce")).groupby(["codigo_xm", "mes"])["v"].mean().rename(columna).reset_index()
    xm_extra = m if xm_extra is None else xm_extra.merge(m, on=["codigo_xm", "mes"], how="outer")

if xm_extra is None:
    xm_extra = pd.DataFrame(columns=["codigo_xm", "mes"])
real = mensual.assign(mes=mensual["mes"].astype(str))[["codigo_xm", "mes", "energia_mwh"]]
xm_extra = xm_extra.merge(real, on=["codigo_xm", "mes"], how="left")
if "gen_prog_despacho_mwh" in xm_extra:
    xm_extra["desvio_despacho_mwh"] = xm_extra["energia_mwh"] - xm_extra["gen_prog_despacho_mwh"]
    xm_extra["cumplimiento_despacho"] = xm_extra["energia_mwh"] / xm_extra["gen_prog_despacho_mwh"].where(xm_extra["gen_prog_despacho_mwh"] > 0)
xm_extra = xm_extra.drop(columns="energia_mwh")
xm_extra.to_csv(PROC / "xm_despacho_mensual.csv", index=False)
print(f"{xm_extra.codigo_xm.nunique()} plantas con datos adicionales de XM")
xm_extra.groupby("codigo_xm").mean(numeric_only=True).round(2)

### 3.2 Ubicación de las plantas
Orden de prioridad:
1. **Tabla base** (`UBICACIONES_BASE`).
2. **OpenStreetMap** (consulta automática a Overpass: plantas solares mapeadas en Colombia), emparejadas por nombre.
3. **Revisión manual:** se genera `plantas_ubicacion_revisar.csv`. Completa ahí `lat` y `lon` de las plantas sin ubicación (búscalas en gem.wiki o Google Earth), guárdalo como `plantas_ubicacion.csv` y vuelve a ejecutar la celda.

In [ ]:
f_osm = RAW / "osm_plantas_solares.json"
SERVIDORES_OVERPASS = ["https://overpass-api.de/api/interpreter",
                       "https://overpass.kumi.systems/api/interpreter",
                       "https://overpass.private.coffee/api/interpreter"]
CABECERAS = {"User-Agent": "UNAL-AnalisisGeoespacial/1.0 (proyecto academico; plantas solares Colombia)",
             "Accept": "application/json"}
if DESCARGAR["ubicaciones_osm"] and not f_osm.exists():
    q = """[out:json][timeout:180];
    area["ISO3166-1"="CO"][admin_level=2]->.co;
    (nwr["power"="plant"]["plant:source"="solar"](area.co);
     nwr["power"="plant"]["generator:source"="solar"](area.co););
    out center tags;"""
    for servidor in SERVIDORES_OVERPASS:
        try:
            r = requests.post(servidor, data={"data": q}, headers=CABECERAS, timeout=240)
            r.raise_for_status()
            r.json()                                   # verifica que sea JSON válido
            f_osm.write_text(r.text, encoding="utf-8")
            print("OSM descargado de", servidor)
            break
        except Exception as e:
            print(f"   {servidor} falló: {e}")
            time.sleep(5)
    else:
        print("⚠️ Ningún servidor de OpenStreetMap respondió: se sigue solo con la tabla base y tu archivo manual.")
osm = pd.DataFrame([{"nombre_osm": e.get("tags", {}).get("name", ""),
                     "lat": e.get("lat", e.get("center", {}).get("lat")),
                     "lon": e.get("lon", e.get("center", {}).get("lon")),
                     "potencia_osm": e.get("tags", {}).get("plant:output:electricity", "")}
                    for e in json.loads(f_osm.read_text(encoding="utf-8"))["elements"]]) if f_osm.exists() else pd.DataFrame()
osm = osm[osm.nombre_osm.str.len() > 0] if len(osm) else osm
print(f"OSM: {len(osm)} plantas solares con nombre en Colombia")

f_manual = BASE / "plantas_ubicacion.csv"
filas = []
for _, r in rec_sol[rec_sol[C_COD].isin(mensual.codigo_xm.unique())].iterrows():
    nom, nn = r[C_NOM], normalizar(r[C_NOM])
    fila = {"codigo_xm": r[C_COD], "nombre_xm": nom, "lat": np.nan, "lon": np.nan, "fuente": "", "similitud": np.nan}
    base = [u for u in UBICACIONES_BASE if normalizar(u["clave"]) in nn]
    if base:
        fila.update(lat=base[0]["lat"], lon=base[0]["lon"], fuente=f"tabla base ({base[0]['precision']})", similitud=1.0)
    elif len(osm):
        sims = osm.nombre_osm.map(lambda o: difflib.SequenceMatcher(None, nn, normalizar(o)).ratio())
        i = sims.idxmax()
        if sims[i] >= 0.6:
            fila.update(lat=osm.at[i, "lat"], lon=osm.at[i, "lon"], fuente=f"OSM: {osm.at[i, 'nombre_osm']}",
                        similitud=round(sims[i], 2))
    filas.append(fila)
ubic = pd.DataFrame(filas)
if f_manual.exists():                                           # tus correcciones manuales mandan
    man = pd.read_csv(f_manual, dtype={"codigo_xm": str}).dropna(subset=["lat", "lon"])
    ubic = ubic.set_index("codigo_xm"); ubic.update(man.set_index("codigo_xm")[["lat", "lon"]])
    ubic.loc[ubic.index.isin(man.codigo_xm), "fuente"] = "manual"; ubic = ubic.reset_index()
ubic.to_csv(BASE / "plantas_ubicacion_revisar.csv", index=False)
print(f"Ubicadas: {ubic.lat.notna().sum()} de {len(ubic)} → revisa plantas_ubicacion_revisar.csv")
ubic.sort_values("fuente")

In [ ]:
plantas = gpd.GeoDataFrame(ubic.dropna(subset=["lat", "lon"]),
                           geometry=gpd.points_from_xy(ubic.dropna(subset=["lat", "lon"]).lon,
                                                       ubic.dropna(subset=["lat", "lon"]).lat), crs=4326)
# Atributos de área por superposición (punto en polígono)
plantas = gpd.sjoin(plantas, dptos[["shapeName", "geometry"]].rename(columns={"shapeName": "departamento"}),
                    how="left", predicate="within").drop(columns="index_right")
plantas = gpd.sjoin(plantas, regimen[["regimen", "geometry"]], how="left", predicate="within").drop(columns="index_right")
plantas = gpd.sjoin(plantas, zrad[["rango", "rad_kwh_m2_dia", "geometry"]], how="left", predicate="within") \
             .drop(columns="index_right").drop_duplicates("codigo_xm")
plantas.to_file(PROC / "plantas_solares.gpkg", driver="GPKG")
plantas.drop(columns="geometry")

### 3.3 Nubosidad, radiación y lluvia mensual en cada planta (NASA POWER)
API pública, sin cuenta (comunidad `RE`, energías renovables). Variables:
| Parámetro | Qué es | Unidad |
|---|---|---|
| `CLOUD_AMT` | Cobertura de nubes | % |
| `ALLSKY_SFC_SW_DWN` | Radiación global en superficie (cielo real) | kWh/m²·día |
| `CLRSKY_SFC_SW_DWN` | Radiación en cielo despejado | kWh/m²·día |
| `ALLSKY_KT` | Índice de claridad | 0–1 |
| `PRECTOTCORR` | Precipitación | mm/día |
| `T2M` | Temperatura a 2 m | °C |

In [ ]:
PARAMS = ["CLOUD_AMT", "ALLSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_DWN", "ALLSKY_KT", "PRECTOTCORR", "T2M"]
f_power = PROC / f"power_mensual_plantas_{PERIODO}.csv"
if DESCARGAR["power"] and not f_power.exists():
    filas = []
    for _, p in plantas.iterrows():
        url = (f"https://power.larc.nasa.gov/api/temporal/monthly/point?parameters={','.join(PARAMS)}"
               f"&community=RE&longitude={p.lon:.4f}&latitude={p.lat:.4f}&start={INICIO.year}&end={FIN.year}&format=JSON")
        par = get(url).json()["properties"]["parameter"]
        for clave in par[PARAMS[0]]:
            if clave.endswith("13"):                 # el mes "13" es el promedio anual
                continue
            filas.append({"codigo_xm": p.codigo_xm, "mes": f"{clave[:4]}-{clave[4:]}",
                          **{k: par[k].get(clave) for k in PARAMS}})
        time.sleep(1)
    pw = pd.DataFrame(filas).replace(-999, np.nan)
    pw.to_csv(f_power, index=False)
pw = pd.read_csv(f_power, dtype={"codigo_xm": str})
pw.groupby("codigo_xm")[PARAMS].mean().round(2)

### 3.4 Estaciones del IDEAM (notebook aparte)
La descarga de las series de las estaciones del IDEAM (datos.gov.co) es lenta, así que está en **`descarga_estaciones_ideam.ipynb`**. Ejecútalo después de las secciones 1–3 de este notebook (necesita `plantas_solares.gpkg`). Si sus resultados existen, la sección 5 los incluye automáticamente en la tabla panel.

### 3.5 Normales climatológicas del IDEAM 1981–2010 (promedios mensuales por estación)
Archivo oficial del IDEAM **«Promedios Climatológicos 1981–2010»** (Excel). Cópialo en `datos/crudos/ideam_normales_1981_2010.xlsx`, o déjalo junto al notebook con su nombre original. Trae, para cada estación, el valor medio de cada mes en 30 años:

| Hoja | Variable | Estaciones (aprox.) |
|---|---|---|
| Lluvia | Precipitación (mm) y número de días con lluvia | ~2.000 |
| Temperatura | Media, máxima media y mínima media (°C) | ~450 |
| **Brillo Solar** | **Horas de sol por día** | ~340 |
| Humedad Relativa | % | ~450 |
| Evaporación / Evapotranspiración potencial | mm | ~400 / ~1.950 |

La celda convierte las coordenadas de grados-minutos-segundos a decimales y lee las 6 hojas. Luego, a cada planta le asigna la **estación más cercana** con cada variable (a ≤ `RADIO_NORMALES_KM`), con su distancia. El **brillo solar** también se **interpola (IDW)**, para tener un valor aunque no haya estación cerca.

**Para qué sirven:**
- **Brillo solar:** es la medida en tierra más directa del efecto de las nubes sobre el sol.
- **Línea base de cada planta:** cómo es un mes «normal» en cada sitio. En la tabla final se calculan las **anomalías** del periodo frente a lo normal (lluvia y días de lluvia con CHIRPS; temperatura con la estación del IDEAM o, si no hay, con NASA POWER), que conectan con El Niño 2023–2024.
- **Datos tipo punto para interpolar:** kriging o IDW de brillo solar o lluvia en todo el país (mapa 8).

In [ ]:
import re, unicodedata
F_NORMALES = RAW / "ideam_normales_1981_2010.xlsx"     # archivo del IDEAM "Promedios Climatológicos 1981-2010"
if not F_NORMALES.exists():                             # también lo busca junto al notebook con su nombre original
    candidatos = sorted(BASE.glob("*romedios*limatol*.xlsx"))
    if candidatos:
        F_NORMALES = candidatos[0]
RADIO_NORMALES_KM = 50
MESES_ES = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]
VARIABLE_POR_TITULO = [            # (texto del título del bloque sin espacios ni tildes, nombre corto)
    ("DIASCONLLUVIA", "dias_lluvia"), ("PRECIPITACION", "ppt_mm"),
    ("TEMPERATURAMAXIMA", "temp_max_c"), ("TEMPERATURAMINIMA", "temp_min_c"), ("TEMPERATURAMEDIA", "temp_media_c"),
    ("BRILLOSOLAR", "brillo_h_dia"), ("HUMEDADRELATIVA", "hr_pct"),
    ("EVAPOTRANSPIRACION", "etp_mm"), ("EVAPORACION", "evap_mm"),
]

def compacto(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"\s+", "", t)

def dms_a_decimal(t):
    """73°12'34.0"W → -73.2094 ; acepta grados y minutos sin segundos."""
    if pd.isna(t):
        return np.nan
    if isinstance(t, (int, float)):
        return float(t)
    t = str(t).replace(",", ".")                      # algunas hojas usan coma decimal
    m = re.match(r"\s*(\d+(?:\.\d+)?)\s*°\s*(?:(\d+(?:\.\d+)?)\s*['’′])?\s*(?:(\d+(?:\.\d+)?)\s*[\"”″]|'')?\s*([NSEWO])", t)
    if not m:
        return np.nan
    g, mi, s, h = m.groups()
    v = float(g) + float(mi or 0) / 60 + float(s or 0) / 3600
    return -v if h in "SWO" else v

def leer_hoja_normales(hoja):
    crudo = pd.read_excel(F_NORMALES, sheet_name=hoja, header=None)
    titulos, encabezado = crudo.iloc[0], crudo.iloc[1].astype(str).str.strip()
    datos = crudo.iloc[2:].reset_index(drop=True)
    est = pd.DataFrame({
        "codigo": datos[0].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10),
        "categoria": datos[1], "nombre": datos[2], "municipio": datos[3], "departamento": datos[4],
        "elevacion_m": pd.to_numeric(datos[5], errors="coerce"),
        "lon": datos[6].map(dms_a_decimal), "lat": datos[7].map(dms_a_decimal)})
    filas, titulo = [], None
    for j in range(8, crudo.shape[1]):
        if pd.notna(titulos[j]):
            titulo = compacto(titulos[j])
        var = next((v for k, v in VARIABLE_POR_TITULO if titulo and k in titulo), None)
        etiqueta = encabezado[j].upper()
        if var is None or etiqueta not in MESES_ES:
            continue
        filas.append(est.assign(variable=var, mes_del_año=MESES_ES.index(etiqueta) + 1,
                                valor=pd.to_numeric(datos[j], errors="coerce")))
    return pd.concat(filas, ignore_index=True) if filas else pd.DataFrame()

if F_NORMALES.exists():
    normales = pd.concat([leer_hoja_normales(h) for h in pd.ExcelFile(F_NORMALES).sheet_names], ignore_index=True)
    normales = normales.dropna(subset=["lat", "lon", "valor"])
    normales = normales[normales["codigo"].str.fullmatch(r"\d{10}")]
    normales.to_csv(PROC / "ideam_normales_1981_2010.csv", index=False)
    print(normales.groupby("variable").agg(estaciones=("codigo", "nunique"), registros=("valor", "size")))
else:
    normales = pd.DataFrame(columns=["codigo", "variable", "mes_del_año", "valor", "lat", "lon"])
    print("⚠️ No encontré el archivo de normales: cópialo como", RAW / "ideam_normales_1981_2010.xlsx")

In [ ]:
def distancia_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def idw(lon_e, lat_e, val, lon_q, lat_q, potencia=2, vecinos=8):
    """Interpolación por distancia inversa (IDW) con las estaciones más cercanas."""
    lon_q, lat_q = np.atleast_1d(lon_q), np.atleast_1d(lat_q)
    kx = 111.32 * np.cos(np.radians(lat_q))[:, None]
    d = np.hypot((lon_q[:, None] - lon_e[None, :]) * kx, (lat_q[:, None] - lat_e[None, :]) * 110.57)
    idx = np.argsort(d, axis=1)[:, :vecinos]
    dk = np.maximum(np.take_along_axis(d, idx, axis=1), 0.1)
    w = 1 / dk ** potencia
    return (w * val[idx]).sum(axis=1) / w.sum(axis=1)

# Para cada planta y variable: la estación más cercana con normales (≤ RADIO_NORMALES_KM) y su distancia
filas = []
for var, d in normales.groupby("variable"):
    est = d.drop_duplicates("codigo")[["codigo", "lat", "lon"]].reset_index(drop=True)
    for _, p in plantas.iterrows():
        dist = distancia_km(p.lat, p.lon, est.lat.values, est.lon.values)
        k = int(np.argmin(dist))
        if dist[k] <= RADIO_NORMALES_KM:
            v = d[d.codigo == est.codigo[k]][["mes_del_año", "valor"]]
            filas.append(v.assign(codigo_xm=p.codigo_xm, variable=var, estacion=est.codigo[k], dist_km=round(float(dist[k]), 1)))
    # brillo solar también interpolado (IDW), para plantas sin estación cercana
    if var == "brillo_h_dia":
        for m, dm in d.groupby("mes_del_año"):
            v = idw(dm.lon.values, dm.lat.values, dm.valor.values, plantas.lon.values, plantas.lat.values)
            filas.append(pd.DataFrame({"codigo_xm": plantas.codigo_xm.values, "mes_del_año": m, "valor": v,
                                       "variable": "brillo_idw_h_dia", "estacion": "IDW", "dist_km": np.nan}))
norm_pl = pd.concat(filas, ignore_index=True) if filas else pd.DataFrame(
    columns=["codigo_xm", "mes_del_año", "valor", "variable", "estacion", "dist_km"])
norm_pl.to_csv(PROC / "ideam_normales_por_planta.csv", index=False)
normales_planta = norm_pl.pivot_table(index=["codigo_xm", "mes_del_año"], columns="variable", values="valor").add_prefix("normal_")
dist_planta = (norm_pl[norm_pl.estacion != "IDW"].drop_duplicates(["codigo_xm", "variable"])
               .pivot(index="codigo_xm", columns="variable", values="dist_km").add_prefix("normal_dist_km_"))
normales_planta = normales_planta.reset_index().merge(dist_planta.reset_index(), on="codigo_xm", how="left")
print("Distancia (km) de cada planta a la estación con normales más cercana:")
dist_planta.round(1)

## 4. Datos tipo SUPERFICIE
### 4.1 Precipitación mensual CHIRPS v2.0 (0,05°)
Recorte de Colombia vía el servidor ERDDAP de NOAA CoastWatch (sin cuenta), **un año por consulta**. Cada año se guarda en `datos/crudos/chirps_mensual/`, así que si algo falla al volver a correr solo pide lo que falta.

Si ERDDAP no responde para un año, ese año se baja mes a mes de los GeoTIFF oficiales de UCSB. Cada archivo es global (~15–20 MB), se descarga con `requests` y se recorta a Colombia en memoria.

In [ ]:
import gzip
import xarray as xr
import rasterio
from rasterio.io import MemoryFile
from rasterio.windows import from_bounds

f_chirps = RAW / f"chirps_mensual_colombia_{PERIODO}.nc"
carp_ch = RAW / "chirps_mensual"; carp_ch.mkdir(parents=True, exist_ok=True)
lo0, la0, lo1, la1 = BBOX_COL

def chirps_erddap(t0, t1):
    url = ("https://coastwatch.pfeg.noaa.gov/erddap/griddap/chirps20GlobalMonthlyP05.nc?"
           f"precip[({t0:%Y-%m-01}T00:00:00Z):1:({t1:%Y-%m-01}T00:00:00Z)][({la0}):1:({la1})][({lo0}):1:({lo1})]")
    r = requests.get(url, timeout=900)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    return r.content

def chirps_ucsb_mes(m):
    u = f"https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/tifs/chirps-v2.0.{m.year}.{m.month:02d}.tif.gz"
    r = requests.get(u, timeout=900); r.raise_for_status()
    with MemoryFile(gzip.decompress(r.content)) as mf, mf.open() as src:
        w = from_bounds(lo0, la0, lo1, la1, src.transform)
        a = src.read(1, window=w).astype("float32"); a[a < 0] = np.nan
        t = src.window_transform(w)
    lats = t.f + t.e * (np.arange(a.shape[0]) + 0.5); lons = t.c + t.a * (np.arange(a.shape[1]) + 0.5)
    return xr.DataArray(a[None], dims=("time", "latitude", "longitude"),
                        coords={"time": [m.to_timestamp()], "latitude": lats, "longitude": lons}, name="precip")

def normaliza(p):
    p = p.rename({k: v for k, v in {"lat": "latitude", "lon": "longitude"}.items() if k in p.dims})
    return p.sortby("latitude")

if DESCARGAR["chirps"] and not f_chirps.exists():
    partes = []
    for a in range(INICIO.year, FIN.year + 1):
        t0, t1 = max(INICIO, date(a, 1, 1)), min(FIN, date(a, 12, 31))
        f_a = carp_ch / f"chirps_{a}_{t0:%m}-{t1:%m}.nc"
        if not f_a.exists():
            try:
                f_a.write_bytes(chirps_erddap(t0, t1)); print(f"CHIRPS {a}: ERDDAP ok")
            except Exception as e:
                print(f"CHIRPS {a}: ERDDAP falló ({str(e)[:150]}) → GeoTIFF de UCSB mes a mes")
                capas = []
                for m in meses(t0, t1):
                    try:
                        capas.append(chirps_ucsb_mes(m))
                    except Exception as e2:
                        print(f"   UCSB {m} falló: {str(e2)[:150]}")
                if len(capas) == len(meses(t0, t1)):
                    xr.concat(capas, dim="time").to_dataset().to_netcdf(f_a)        # año completo: se guarda
                elif capas:
                    partes.append(normaliza(xr.concat(capas, dim="time")))           # incompleto: se usa, pero se reintentará
        if f_a.exists():
            with xr.open_dataset(f_a) as ds:
                partes.append(normaliza(ds["precip"].load()))
    if not partes:
        raise RuntimeError("No se pudo descargar CHIRPS de ninguna fuente: revisa la conexión a internet y vuelve a ejecutar")
    ref = partes[0]                                    # misma grilla para todos los años (ERDDAP y UCSB difieren en milésimas)
    partes = [p.reindex(latitude=ref.latitude, longitude=ref.longitude, method="nearest", tolerance=0.02) for p in partes]
    comb = xr.concat(partes, dim="time").sortby("time").to_dataset(name="precip")
    faltan = len(meses(INICIO, FIN)) - comb.sizes["time"]
    if faltan == 0:
        comb.to_netcdf(f_chirps)
    else:
        print(f"⚠️ Faltan {faltan} meses de CHIRPS: se sigue con los disponibles; vuelve a ejecutar la celda más tarde para completarlos")
        chirps = comb
if f_chirps.exists():
    chirps = xr.open_dataset(f_chirps)
print(chirps["precip"].shape, "· meses:", chirps.sizes["time"])

### 4.2 Precipitación diaria CHIRPS en cada planta: días de lluvia
La lluvia mensual no dice si llovió un día fuerte o veinte días seguidos; para la generación solar importa más lo segundo. Por eso se descarga la **serie diaria** de CHIRPS en el píxel de cada planta (ERDDAP de NOAA, conjunto `chirps20GlobalDailyP05`, solo un píxel por planta: archivos muy pequeños) y se calcula por mes:
- `chirps_dias_lluvia`: días con lluvia ≥ `UMBRAL_LLUVIA_MM`,
- `chirps_ppt_max_diaria_mm`: el día más lluvioso,
- `chirps_ppt_diaria_mm`: suma de los días (sirve para comparar con el CHIRPS mensual).

In [ ]:
import io
f_chd = RAW / f"chirps_diario_plantas_{PERIODO}.csv"
URL_CHD = ("https://coastwatch.pfeg.noaa.gov/erddap/griddap/chirps20GlobalDailyP05.csv?"
           "precip%5B({t0}T00:00:00Z):1:({t1}T00:00:00Z)%5D%5B({lat:.3f})%5D%5B({lon:.3f})%5D")
if DESCARGAR["chirps_diario"] and not f_chd.exists():
    partes = []
    for _, p in plantas.iterrows():
        try:
            txt = get(URL_CHD.format(t0=f"{INICIO:%Y-%m-%d}", t1=f"{FIN:%Y-%m-%d}", lat=p.lat, lon=p.lon), timeout=300).text
            d = pd.read_csv(io.StringIO(txt), skiprows=[1])            # la 2.ª fila son las unidades
            partes.append(d.assign(codigo_xm=p.codigo_xm))
        except Exception as e:
            print("   CHIRPS diario falló para", p.codigo_xm, ":", e)
        time.sleep(1)
    pd.concat(partes, ignore_index=True).to_csv(f_chd, index=False)
chd = pd.read_csv(f_chd, dtype={"codigo_xm": str})
chd["precip"] = pd.to_numeric(chd["precip"], errors="coerce")
chd["mes"] = pd.to_datetime(chd["time"]).dt.tz_localize(None).dt.to_period("M").astype(str)
chd_mes = (chd.groupby(["codigo_xm", "mes"])["precip"]
              .agg(chirps_ppt_diaria_mm="sum", chirps_ppt_max_diaria_mm="max",
                   chirps_dias_lluvia=lambda v: int((v >= UMBRAL_LLUVIA_MM).sum()))
              .reset_index())
chd_mes.to_csv(PROC / "chirps_diario_mensual_plantas.csv", index=False)
chd_mes.groupby("codigo_xm")[["chirps_ppt_diaria_mm", "chirps_dias_lluvia"]].mean().round(1)

### 4.3 Frecuencia de nubes MODIS · EarthEnv (~1 km)
Climatología: porcentaje de días nublados de cada mes del año (2000–2014), más el promedio anual (Wilson y Jetz, 2016). Muestra **dónde** es más nublado con mucho detalle espacial. Se recorta a Colombia leyendo solo esa ventana del archivo remoto.

In [ ]:
import rasterio
from rasterio.windows import from_bounds
carp_nubes = RAW / "earthenv_nubes"; carp_nubes.mkdir(parents=True, exist_ok=True)
archivos_nubes = {f"{m:02d}": f"MODCF_monthlymean_{m:02d}.tif" for m in range(1, 13)} | {"anual": "MODCF_meanannual.tif"}
if DESCARGAR["nubes_earthenv"]:
    for clave, nombre in archivos_nubes.items():
        destino = carp_nubes / f"nubes_{clave}_colombia.tif"
        if destino.exists():
            continue
        with rasterio.open(f"/vsicurl/https://data.earthenv.org/cloud/{nombre}") as src:
            w = from_bounds(*BBOX_COL, src.transform)
            a = src.read(1, window=w)
            perfil = src.profile | {"height": a.shape[0], "width": a.shape[1], "transform": src.window_transform(w),
                                    "driver": "GTiff", "compress": "lzw"}
        with rasterio.open(destino, "w", **perfil) as dst:
            dst.write(a, 1)
        print("ok", destino.name)
with rasterio.open(carp_nubes / "nubes_anual_colombia.tif") as src:
    print("Valores crudos de ejemplo:", np.unique(src.read(1)[::200, ::200])[:10],
          "→ EarthEnv guarda % × 100 (divide entre 100)")

### 4.4 Elevación en cada planta (Copernicus GLO-30)
Se lee solo el píxel de cada planta en la tesela remota.

In [ ]:
URL_DEM = ("/vsicurl/https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM/"
           "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM.tif")
if DESCARGAR["elevacion"] and "elevacion_m" not in plantas.columns:
    elev = []
    for _, p in plantas.iterrows():
        la, lo = int(np.floor(p.lat)), int(np.floor(p.lon))
        u = URL_DEM.format(ns="N" if la >= 0 else "S", lat=abs(la), ew="W" if lo < 0 else "E", lon=abs(lo))
        try:
            with rasterio.open(u) as src:
                elev.append(float(next(src.sample([(p.lon, p.lat)]))[0]))
        except Exception:
            elev.append(np.nan)
    plantas["elevacion_m"] = elev
    plantas.to_file(PROC / "plantas_solares.gpkg", driver="GPKG")
plantas[["nombre_xm", "departamento", "regimen", "rango", "elevacion_m"]]

## 5. Tabla panel planta × mes (lista para el análisis)
Une todo en una fila por planta y mes:
- **Variable respuesta:** factor de planta (y el desvío entre generación real y programada).
- **Variables explicativas:**
  - nubosidad y radiación (POWER), irradiación y temperatura medidas en la planta (XM),
  - despacho ideal, generación programada y disponibilidad (XM),
  - lluvia (POWER, CHIRPS mensual y diario en el píxel de la planta: total, días de lluvia, día máximo),
  - estaciones del IDEAM más cercanas con dato ese mes: lluvia, días de lluvia, temperatura y humedad (y su distancia),
  - frecuencia de nubes de EarthEnv del mismo mes del año,
  - normales IDEAM 1981–2010 del mismo mes (lluvia, días de lluvia, temperatura, brillo solar, humedad) y **anomalías** del periodo frente a ellas,
  - régimen de lluvias, zona de radiación, departamento y elevación.

In [ ]:
panel = mensual.copy()
panel["mes"] = panel["mes"].astype(str)
panel = panel.merge(pw, on=["codigo_xm", "mes"], how="left")
panel = panel.merge(xm_extra, on=["codigo_xm", "mes"], how="left")      # despacho y programada (XM)
panel = panel.merge(plantas.drop(columns="geometry")[["codigo_xm", "nombre_xm", "lat", "lon", "departamento",
                                                      "regimen", "rango", "rad_kwh_m2_dia", "elevacion_m"]],
                    on="codigo_xm", how="inner")
# CHIRPS en el píxel de cada planta
ch = chirps["precip"]
lat_n = "latitude" if "latitude" in ch.dims else "lat"; lon_n = "longitude" if "longitude" in ch.dims else "lon"
vals = []
for _, p in plantas.iterrows():
    s = ch.sel({lat_n: p.lat, lon_n: p.lon}, method="nearest").to_series()
    s.index = pd.PeriodIndex(s.index, freq="M").astype(str)
    vals.append(s.rename("chirps_mm_mes").to_frame().assign(codigo_xm=p.codigo_xm))
panel = panel.merge(pd.concat(vals).reset_index(names="mes"), on=["codigo_xm", "mes"], how="left")
# EarthEnv: frecuencia de nubes del mismo mes del año
nub = {}
for m in range(1, 13):
    with rasterio.open(RAW / "earthenv_nubes" / f"nubes_{m:02d}_colombia.tif") as src:
        nub[m] = [float(v[0]) / 100 for v in src.sample(list(zip(plantas.lon, plantas.lat)))]
nub = pd.DataFrame(nub, index=plantas.codigo_xm.values).stack().rename("nubes_earthenv_pct").reset_index()
nub.columns = ["codigo_xm", "mes_del_año", "nubes_earthenv_pct"]
panel["mes_del_año"] = pd.PeriodIndex(panel["mes"], freq="M").month
panel = panel.merge(nub, on=["codigo_xm", "mes_del_año"], how="left")
# CHIRPS diario: días de lluvia y día más lluvioso de cada mes
panel = panel.merge(chd_mes, on=["codigo_xm", "mes"], how="left")
# IDEAM (notebook aparte): para cada planta y mes, la estación más cercana que tenga dato de cada variable
f_est, f_estpl = PROC / "ideam_estaciones_mensual.csv", PROC / "ideam_estaciones_por_planta.csv"
if f_est.exists() and f_estpl.exists():
    est_mes = pd.read_csv(f_est, dtype={"codigoestacion": str})
    est_plantas = pd.read_csv(f_estpl, dtype={"codigo": str, "codigo_xm": str})
    cerc = est_plantas[["codigo_xm", "codigo", "distancia_km"]].merge(
        est_mes, left_on="codigo", right_on="codigoestacion").sort_values("distancia_km")
    for cols, sufijo in ((["ideam_ppt_mm", "ideam_dias_lluvia", "ideam_ppt_max_diaria_mm"], "ppt"),
                         (["ideam_temp_media_c", "ideam_temp_max_c", "ideam_temp_min_c"], "temp"),
                         (["ideam_hr_media_pct"], "hr")):
        sel = (cerc.dropna(subset=[cols[0]]).groupby(["codigo_xm", "mes"]).first().reset_index()
                   [["codigo_xm", "mes", *cols, "codigo", "distancia_km"]]
                   .rename(columns={"codigo": f"ideam_est_{sufijo}", "distancia_km": f"ideam_dist_{sufijo}_km"}))
        panel = panel.merge(sel, on=["codigo_xm", "mes"], how="left")
else:
    print("Sin datos de estaciones IDEAM (opcional): ejecuta descarga_estaciones_ideam.ipynb y vuelve a correr esta celda.")
# Normales IDEAM 1981–2010 del mismo mes y anomalías del periodo
panel = panel.merge(normales_planta, on=["codigo_xm", "mes_del_año"], how="left")
if "normal_ppt_mm" in panel:
    panel["anom_ppt_mm"] = panel["chirps_mm_mes"] - panel["normal_ppt_mm"]          # en meses secos usa esta, no el %
    panel["anom_ppt_pct"] = 100 * (panel["chirps_mm_mes"] / panel["normal_ppt_mm"].where(panel["normal_ppt_mm"] > 0) - 1)
if "normal_dias_lluvia" in panel:
    panel["anom_dias_lluvia"] = panel["chirps_dias_lluvia"] - panel["normal_dias_lluvia"]
if "normal_temp_media_c" in panel:
    t_obs = panel["ideam_temp_media_c"].fillna(panel["T2M"]) if "ideam_temp_media_c" in panel else panel["T2M"]
    panel["anom_temp_c"] = t_obs - panel["normal_temp_media_c"]
panel.to_csv(PROC / "panel_plantas_mes.csv", index=False)
print(f"Panel: {panel.codigo_xm.nunique()} plantas × {panel.mes.nunique()} meses = {len(panel)} filas")
panel.head()

## 6. Inventario y mapa de verificación

In [ ]:
for f in sorted(PROC.glob("*")):
    print(f"{f.name:45s} {f.stat().st_size / 1e6:8.2f} MB")

fig, axs = plt.subplots(1, 3, figsize=(21, 8))
regimen.plot(ax=axs[0], column="regimen", cmap="Set3", legend=True, legend_kwds={"fontsize": 7, "loc": "lower left"})
dptos.boundary.plot(ax=axs[0], color="0.4", linewidth=0.3)
plantas.plot(ax=axs[0], color="k", marker="^", markersize=60)
axs[0].set_title("Régimen pluviométrico (IDEAM) y plantas solares", loc="left", fontweight="bold")

with rasterio.open(RAW / "earthenv_nubes" / "nubes_anual_colombia.tif") as src:
    a = src.read(1).astype(float) / 100; b = src.bounds
im = axs[1].imshow(np.where(a > 0, a, np.nan), extent=[b.left, b.right, b.bottom, b.top], cmap="Blues")
plt.colorbar(im, ax=axs[1], shrink=0.6, label="% de días nublados")
dptos.boundary.plot(ax=axs[1], color="0.3", linewidth=0.3); plantas.plot(ax=axs[1], color="#E0A526", marker="^", markersize=60, edgecolor="k")
axs[1].set_title("Frecuencia anual de nubes (EarthEnv/MODIS)", loc="left", fontweight="bold")

im = chirps["precip"].sum("time").plot(ax=axs[2], cmap="YlGnBu", add_colorbar=False)
plt.colorbar(im, ax=axs[2], shrink=0.6, label="mm en el periodo")
dptos.boundary.plot(ax=axs[2], color="0.3", linewidth=0.3); plantas.plot(ax=axs[2], color="#E0A526", marker="^", markersize=60, edgecolor="k")
axs[2].set_title("Precipitación acumulada CHIRPS", loc="left", fontweight="bold")
for ax in axs:
    ax.set_xlim(BBOX_COL[0], BBOX_COL[2]); ax.set_ylim(BBOX_COL[1], BBOX_COL[3])
plt.tight_layout(); plt.savefig(OUT / "verificacion_datos_v2.png", dpi=150); plt.show()

In [ ]:
# Primera mirada (no es el análisis): factor de planta mensual vs. nubosidad POWER
fig, ax = plt.subplots(figsize=(8, 6))
for reg, d in panel.dropna(subset=["factor_planta", "CLOUD_AMT"]).groupby("regimen"):
    ax.scatter(d.CLOUD_AMT, d.factor_planta, s=14, alpha=0.6, label=reg)
ax.set_xlabel("Nubosidad NASA POWER (%)"); ax.set_ylabel("Factor de planta mensual")
ax.legend(fontsize=7); ax.set_title("Exploración inicial por régimen de lluvias", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Mapas para la presentación
Exporta **7 mapas**, uno o dos por tipo de dato, con el mismo estilo y la proporción exacta de los espacios de la diapositiva "Datos espaciales" (vertical, 4 × 5,8 pulgadas, 200 dpi).

| Archivo | Tipo | Contenido |
|---|---|---|
| `01_plantas_factor_planta.png` | Punto | Plantas: tamaño = capacidad, color = factor de planta medio |
| `02_nubosidad_plantas.png` | Punto | Nubosidad media NASA POWER en cada planta |
| `03_regimen_pluviometrico.png` | Área | Régimen de lluvias del IDEAM |
| `04_zonas_radiacion.png` | Área | Zonas de radiación solar del IDEAM |
| `05_precipitacion_chirps.png` | Superficie | Precipitación media anual CHIRPS |
| `06_nubes_modis.png` | Superficie | Frecuencia anual de nubes (EarthEnv/MODIS) |
| `07_brillo_solar_idw.png` | Superficie (interpolada) | Brillo solar normal 1981–2010 (IDW desde ~340 estaciones IDEAM) |

Quedan en `resultados/mapas_presentacion/`. **Súbelos al chat** y los pongo en la diapositiva.

In [ ]:
from rasterio.features import geometry_mask
MAPAS = OUT / "mapas_presentacion"; MAPAS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
lo0, la0, lo1, la1 = BBOX_COL
FIG = (4, 5.8)
colombia = dptos.dissolve()

def base(ax, relleno=False):
    if relleno:
        dptos.plot(ax=ax, color="#F2F2EF", edgecolor="none", zorder=0)
    dptos.boundary.plot(ax=ax, color="0.45", linewidth=0.3, zorder=2)
    ax.set_xlim(lo0, lo1); ax.set_ylim(la0, la1); ax.set_aspect("equal"); ax.set_axis_off()

def barra(fig, ax, m, etiqueta):
    cb = fig.colorbar(m, ax=ax, orientation="horizontal", fraction=0.04, pad=0.01, aspect=30)
    cb.set_label(etiqueta, fontsize=10); cb.ax.tick_params(labelsize=8)

def guardar(fig, nombre):
    fig.savefig(MAPAS / nombre, dpi=200, bbox_inches="tight", facecolor="white"); plt.show()

resumen = panel.groupby("codigo_xm").agg(fp=("factor_planta", "mean"), cap=("capacidad_mw", "first"),
                                         nubes=("CLOUD_AMT", "mean")).reset_index()
pl = plantas.merge(resumen, on="codigo_xm")

# 1 · Plantas y factor de planta
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
m = ax.scatter(pl.geometry.x, pl.geometry.y, s=15 + 1.5 * pl.cap, c=100 * pl.fp, cmap="YlOrRd",
               edgecolor="k", linewidth=0.5, zorder=3)
barra(fig, ax, m, "Factor de planta medio (%) · tamaño = capacidad"); guardar(fig, "01_plantas_factor_planta.png")

# 2 · Nubosidad en cada planta
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
m = ax.scatter(pl.geometry.x, pl.geometry.y, s=70, c=pl.nubes, cmap="Blues", edgecolor="k", linewidth=0.5, zorder=3)
barra(fig, ax, m, "Nubosidad media NASA POWER (%)"); guardar(fig, "02_nubosidad_plantas.png")

# 3 · Régimen pluviométrico
fig, ax = plt.subplots(figsize=FIG)
regimen.plot(ax=ax, column="regimen", cmap="Set3", legend=True,
             legend_kwds={"loc": "upper center", "bbox_to_anchor": (0.5, -0.01), "fontsize": 7, "frameon": False, "ncol": 1})
base(ax); plantas.plot(ax=ax, color="k", marker="^", markersize=18, zorder=3); guardar(fig, "03_regimen_pluviometrico.png")

# 4 · Zonas de radiación
fig, ax = plt.subplots(figsize=FIG)
zrad.plot(ax=ax, column="rad_kwh_m2_dia", cmap="YlOrBr", edgecolor="none")
base(ax); plantas.plot(ax=ax, color="k", marker="^", markersize=18, zorder=3)
sm = plt.cm.ScalarMappable(cmap="YlOrBr", norm=plt.Normalize(zrad.rad_kwh_m2_dia.min(), zrad.rad_kwh_m2_dia.max()))
barra(fig, ax, sm, "Radiación global (kWh/m²·día)"); guardar(fig, "04_zonas_radiacion.png")

# 5 · Precipitación media anual CHIRPS
anual = chirps["precip"].sum("time") / (chirps.sizes["time"] / 12)
fig, ax = plt.subplots(figsize=FIG)
m = anual.plot(ax=ax, cmap="YlGnBu", add_colorbar=False, vmin=0, vmax=6000)
base(ax); plantas.plot(ax=ax, color="#E0A526", marker="^", markersize=18, edgecolor="k", zorder=3)
barra(fig, ax, m, "Precipitación media anual (mm)"); guardar(fig, "05_precipitacion_chirps.png")

# 6 · Frecuencia anual de nubes (EarthEnv), recortada a Colombia
with rasterio.open(RAW / "earthenv_nubes" / "nubes_anual_colombia.tif") as src:
    a = src.read(1).astype(float) / 100
    fuera = geometry_mask(colombia.geometry, out_shape=a.shape, transform=src.transform)
    b = src.bounds
a[fuera | (a <= 0)] = np.nan
fig, ax = plt.subplots(figsize=FIG)
m = ax.imshow(a, extent=[b.left, b.right, b.bottom, b.top], cmap="Greys", vmin=20, vmax=100)
base(ax); plantas.plot(ax=ax, color="#E0A526", marker="^", markersize=18, edgecolor="k", zorder=3)
barra(fig, ax, m, "Días nublados al año (%)"); guardar(fig, "06_nubes_modis.png")
print("Mapas en", MAPAS)

In [ ]:
# 7 · Brillo solar normal 1981–2010 interpolado (IDW) desde las estaciones del IDEAM
b = normales[normales.variable == "brillo_h_dia"].groupby("codigo")[["lon", "lat", "valor"]].mean()
if len(b):
    paso = 0.1
    gx, gy = np.meshgrid(np.arange(lo0, lo1, paso) + paso / 2, np.arange(la1, la0, -paso) - paso / 2)
    z = idw(b.lon.values, b.lat.values, b.valor.values, gx.ravel(), gy.ravel()).reshape(gx.shape)
    from rasterio.transform import from_origin
    fuera = geometry_mask(colombia.geometry, out_shape=z.shape, transform=from_origin(lo0, la1, paso, paso))
    z[fuera] = np.nan
    fig, ax = plt.subplots(figsize=FIG)
    m = ax.imshow(z, extent=[lo0, lo0 + paso * z.shape[1], la1 - paso * z.shape[0], la1], cmap="YlOrRd", vmin=2, vmax=8)
    base(ax)
    ax.scatter(b.lon, b.lat, s=3, c="k", zorder=3)
    plantas.plot(ax=ax, color="#3D6E8F", marker="^", markersize=22, edgecolor="k", zorder=4)
    barra(fig, ax, m, "Brillo solar normal 1981–2010 (horas de sol/día)"); guardar(fig, "07_brillo_solar_idw.png")

## 8. Desviaciones entre la generación real y la programada (XM)
Compara, hora a hora, la **generación real** (`Gene`) con la **programada en el despacho** (`GeneProgDesp`; cambia `REFERENCIA` a `GeneProgRedesp` para usar el redespacho). Usa los archivos que ya guardaron las secciones 3.1 y 3.1b en `datos/crudos/xm/`, así que **no descarga nada**.

**Definiciones**
- Desviación = real − programada (MWh). Positiva: la planta produjo **más** de lo previsto; negativa: **menos** (típico de días nublados o lluviosos).
- Desviación relativa (%) = desviación ÷ programada.
  - XM deja **vacías las horas sin sol**: si faltan las dos series, la hora cuenta como noche (0); si falta solo una, es un dato faltante y no se usa.
  - **Horaria:** solo se evalúa en las horas en que lo programado es ≥ 10 % de la capacidad, para no inflar el % al amanecer y al atardecer. También se guarda la desviación respecto a la capacidad.
  - **Diaria y mensual:** se suman la energía real y la programada del periodo. Solo entran los días con ≤ 2 horas faltantes y los meses con ≥ 80 % de días válidos.
- Las plantas **sin ubicación** entran en las tablas (con su nombre de XM), pero no en los mapas ni en los resúmenes por región.
- Se marca como **desviación grande** toda la que pase el **15 %** (`UMBRAL_DESV`), en valor absoluto.
- **Región:** región natural del departamento (Caribe, Andina, Pacífica, Orinoquía, Amazonía); también se resume por departamento y por régimen de lluvias.

**Indicadores**, para cada escala (`_h` horaria, `_d` diaria, `_m` mensual):

| Indicador | Qué mide |
|---|---|
| `desv_abs_media_pct` | Tamaño típico de la desviación |
| `sesgo_pct` | Si en promedio la planta produce más (+) o menos (−) de lo programado |
| `desv_max_pct` | La mayor desviación, con su signo |
| `pct_sobre_umbral` | % de horas, días o meses con \|desviación\| > 15 % |
| `nmae_pct` | Error absoluto ponderado por energía |
| `desv_total_mwh` | Desviación acumulada |

**Salidas**
- **Tablas** (`datos/procesados/`): `desviaciones_horarias.csv`, `_diarias.csv`, `_mensuales.csv`, `desviaciones_resumen_plantas.csv`, `_regiones.csv`, `_departamentos.csv`, `_regimen.csv` y `desviaciones_eventos_sobre_15pct.csv`.
- **Mapas y gráficos** (`resultados/desviaciones/`):
  - D1: desviación diaria media por planta.
  - D2: % de días por encima del 15 %.
  - D3: sesgo mensual.
  - D4: departamentos según el % de meses por encima del 15 %.
  - G1: ranking de plantas.
  - G2: ciclo hora × mes por región.
  - G3: evolución mensual por región.

### 8.1 Desviaciones horarias, diarias y mensuales

In [ ]:
import re, unicodedata
REFERENCIA = "GeneProgDesp"     # despacho programado; cambia a "GeneProgRedesp" para comparar con el redespacho
UMBRAL_DESV = 0.15              # desviación relativa que se considera grande (15 %)
MIN_PROG_FRAC = 0.10            # una hora se evalúa en % solo si lo programado es ≥ 10 % de la capacidad (evita amaneceres y atardeceres)
CARP_DESV = OUT / "desviaciones"; CARP_DESV.mkdir(parents=True, exist_ok=True)

REGIONES = {   # regiones naturales de Colombia por departamento (texto en mayúsculas y sin tildes)
    "Caribe": ["ATLANTICO", "BOLIVAR", "CESAR", "CORDOBA", "GUAJIRA", "MAGDALENA", "SUCRE", "SAN ANDRES"],
    "Andina": ["ANTIOQUIA", "BOYACA", "CALDAS", "CUNDINAMARCA", "BOGOTA", "HUILA", "SANTANDER", "QUINDIO", "RISARALDA", "TOLIMA"],
    "Pacífica": ["CHOCO", "CAUCA", "NARINO", "VALLE"],
    "Orinoquía": ["ARAUCA", "CASANARE", "META", "VICHADA"],
    "Amazonía": ["AMAZONAS", "CAQUETA", "GUAINIA", "GUAVIARE", "PUTUMAYO", "VAUPES"],
}
def region_de(dpto):
    t = unicodedata.normalize("NFKD", str(dpto)).encode("ascii", "ignore").decode().upper()
    return next((r for r, claves in REGIONES.items() if any(k in t for k in claves)), "Sin región")

def xm_horario_largo(prefijo):
    """Une los archivos mensuales de XM (Hour01…Hour24) en una tabla larga planta × hora, en MWh."""
    patron = re.compile(rf"^{prefijo}_\d{{4}}-\d{{2}}\.csv$")
    archivos = [f for f in sorted((RAW / "xm").glob("*.csv")) if patron.match(f.name)]
    if not archivos:
        raise FileNotFoundError(f"No hay archivos {prefijo}_AAAA-MM.csv en {RAW / 'xm'}: ejecuta antes las secciones 3.1 y 3.1b")
    d = pd.concat([pd.read_csv(f, dtype={"code": str}) for f in archivos], ignore_index=True)
    horas = [c for c in d.columns if c.lower().startswith("hour")]
    l = d.melt(id_vars=["Date", "code"], value_vars=horas, var_name="h", value_name="mwh")
    l["fecha_hora"] = (pd.to_datetime(l["Date"]).dt.tz_localize(None).dt.normalize()
                       + pd.to_timedelta(l["h"].str.extract(r"(\d+)")[0].astype(int) - 1, unit="h"))
    l["mwh"] = pd.to_numeric(l["mwh"], errors="coerce") / 1000          # kWh → MWh
    return l.rename(columns={"code": "codigo_xm"})[["codigo_xm", "fecha_hora", "mwh"]]

real = xm_horario_largo("gene").rename(columns={"mwh": "real_mwh"})
prog = xm_horario_largo(REFERENCIA).rename(columns={"mwh": "prog_mwh"})
cap_mw = pd.read_csv(PROC / "xm_capacidad.csv", dtype={"codigo_xm": str}).set_index("codigo_xm")["capacidad_mw"]

# --- Escala HORARIA
# XM deja vacías las horas sin sol en ambas series: si faltan las dos, es noche (= 0);
# si falta solo una de las dos, es un dato faltante y esa hora no se usa.
h = real.merge(prog, on=["codigo_xm", "fecha_hora"], how="inner")
faltante = h["real_mwh"].isna() ^ h["prog_mwh"].isna()
h[["real_mwh", "prog_mwh"]] = h[["real_mwh", "prog_mwh"]].fillna(0)
h["fecha"] = h["fecha_hora"].dt.normalize()
dias_malos = h.loc[faltante].groupby(["codigo_xm", "fecha"]).size()
dias_malos = set(dias_malos[dias_malos > 2].index)                 # días con más de 2 horas faltantes
h = h[~faltante]
h = h[(h["real_mwh"] > 0) | (h["prog_mwh"] > 0)]                    # solo horas de sol
h["capacidad_mw"] = h["codigo_xm"].map(cap_mw)
h["capacidad_mw"] = h["capacidad_mw"].fillna(h.groupby("codigo_xm")["real_mwh"].transform("max"))   # respaldo
h["desv_mwh"] = h["real_mwh"] - h["prog_mwh"]                              # + produjo más de lo programado, − menos
h["desv_pct"] = (h["desv_mwh"] / h["prog_mwh"]).where(h["prog_mwh"] >= MIN_PROG_FRAC * h["capacidad_mw"])
h["desv_cap_pct"] = h["desv_mwh"] / h["capacidad_mw"]                     # desviación respecto a la capacidad
h["mes"] = h["fecha_hora"].dt.to_period("M").astype(str)
h["hora"] = h["fecha_hora"].dt.hour

# --- Escalas DIARIA y MENSUAL (sumas de energía; el % es sobre lo programado en el periodo)
def agrega(df, clave):
    g = (df.groupby(["codigo_xm", clave])
           .agg(real_mwh=("real_mwh", "sum"), prog_mwh=("prog_mwh", "sum"), horas_sol=("real_mwh", "size"))
           .reset_index())
    g["desv_mwh"] = g["real_mwh"] - g["prog_mwh"]
    g["desv_pct"] = g["desv_mwh"] / g["prog_mwh"].where(g["prog_mwh"] > 0)
    return g

claves_ok = ~pd.Series(list(zip(h.codigo_xm, h.fecha)), index=h.index).isin(dias_malos)
d = agrega(h[claves_ok], "fecha")                                   # días con ≤ 2 horas faltantes
d["mes"] = d["fecha"].dt.to_period("M").astype(str)
dias_validos = d.groupby(["codigo_xm", "mes"]).size().rename("dias_validos").reset_index()
m = agrega(h[claves_ok], "mes").merge(dias_validos, on=["codigo_xm", "mes"])
m = m[m["dias_validos"] >= 0.8 * pd.PeriodIndex(m["mes"], freq="M").days_in_month]   # meses con ≥ 80 % de días válidos

# --- Información de cada planta: nombre (listado de XM) y, si tiene ubicación, departamento, región y régimen
nombres = rec_sol.drop_duplicates(C_COD).set_index(C_COD)[C_NOM]
info = pd.DataFrame({"codigo_xm": sorted(h["codigo_xm"].unique())})
info["nombre_xm"] = info["codigo_xm"].map(nombres).fillna(info["codigo_xm"])
info = info.merge(plantas.drop(columns="geometry")[["codigo_xm", "departamento", "regimen"]].drop_duplicates("codigo_xm"),
                  on="codigo_xm", how="left")
info["region"] = info["departamento"].map(region_de).where(info["departamento"].notna(), "Sin ubicación")
info["capacidad_mw"] = info["codigo_xm"].map(cap_mw)
sin_ubic = (info["region"] == "Sin ubicación").sum()
if sin_ubic:
    print(f"⚠️ {sin_ubic} plantas no tienen ubicación: entran en las tablas pero no en los mapas ni en las regiones "
          "(complétalas en plantas_ubicacion.csv, sección 3.2)")
h, d, m = (t.merge(info.drop(columns="capacidad_mw", errors="ignore"), on="codigo_xm", how="left") for t in (h, d, m))
for t in (h, d, m):
    t["supera_umbral"] = t["desv_pct"].abs() > UMBRAL_DESV

h.to_csv(PROC / "desviaciones_horarias.csv", index=False)
d.to_csv(PROC / "desviaciones_diarias.csv", index=False)
m.to_csv(PROC / "desviaciones_mensuales.csv", index=False)
print(f"Referencia: {REFERENCIA} · {h.codigo_xm.nunique()} plantas con generación real y programada")
print(f"Horas de sol: {len(h):,} ({h.desv_pct.notna().sum():,} evaluables en %) · días válidos: {len(d):,} · meses válidos: {len(m):,}")
if d.empty or m.empty:
    print("⚠️ No quedaron días o meses completos: revisa que existan los archivos gene_ y " + REFERENCIA + "_ de los mismos meses")


### 8.2 Mayores desviaciones por planta y por región, y eventos que pasan el 15 %

In [ ]:
def resumen(df, por, sufijo):
    """Indicadores de desviación agrupando por planta, departamento, región o régimen."""
    v = df.dropna(subset=["desv_pct"])
    g = v.groupby(por)
    r = pd.DataFrame({
        "n": g.size(),
        "desv_abs_media_pct": 100 * g["desv_pct"].apply(lambda s: s.abs().mean()),      # tamaño típico del error
        "sesgo_pct": 100 * g["desv_pct"].mean(),                                         # + sobre-genera, − sub-genera
        "desv_max_pct": 100 * g["desv_pct"].apply(lambda s: s.loc[s.abs().idxmax()]),    # la mayor, con su signo
        "pct_sobre_umbral": 100 * g["supera_umbral"].mean(),                             # % de periodos con |desv| > 15 %
        "nmae_pct": 100 * g["desv_mwh"].apply(lambda s: s.abs().sum()) / g["prog_mwh"].sum(),  # error ponderado por energía
        "desv_total_mwh": g["desv_mwh"].sum(),
    })
    return r.add_suffix(f"_{sufijo}")

escalas = {"h": h, "d": d, "m": m}
por_planta = info.set_index("codigo_xm").join(pd.concat([resumen(t, "codigo_xm", k) for k, t in escalas.items()], axis=1), how="inner")
por_region = pd.concat([resumen(t, "region", k) for k, t in escalas.items()], axis=1)
por_dpto = pd.concat([resumen(t, "departamento", k) for k, t in escalas.items()], axis=1)
por_regimen = pd.concat([resumen(t, "regimen", k) for k, t in escalas.items()], axis=1)
por_planta.to_csv(PROC / "desviaciones_resumen_plantas.csv")
por_region.to_csv(PROC / "desviaciones_resumen_regiones.csv")
por_dpto.to_csv(PROC / "desviaciones_resumen_departamentos.csv")
por_regimen.to_csv(PROC / "desviaciones_resumen_regimen.csv")

# Eventos que pasan el 15 %
eventos = pd.concat([t[t.supera_umbral].assign(escala=k) for k, t in (("diaria", d), ("mensual", m))], ignore_index=True)
eventos.sort_values("desv_pct", key=abs, ascending=False).to_csv(PROC / "desviaciones_eventos_sobre_15pct.csv", index=False)

cols = ["nombre_xm", "region", "capacidad_mw", "desv_abs_media_pct_d", "sesgo_pct_m", "pct_sobre_umbral_h",
        "pct_sobre_umbral_d", "pct_sobre_umbral_m", "desv_max_pct_m"]
print("Plantas ordenadas por % de DÍAS con desviación mayor al 15 %:")
display(por_planta.sort_values("pct_sobre_umbral_d", ascending=False)[cols].round(1))
print("Por región:")
display(por_region[["n_d", "desv_abs_media_pct_d", "sesgo_pct_d", "pct_sobre_umbral_h", "pct_sobre_umbral_d", "pct_sobre_umbral_m", "nmae_pct_m"]].round(1))
print(f"\nMayores desviaciones MENSUALES (|desv| > {UMBRAL_DESV:.0%}): {m.supera_umbral.sum()} de {m.desv_pct.notna().sum()} planta-mes")
display(m[m.supera_umbral].sort_values("desv_pct", key=abs, ascending=False)
          [["nombre_xm", "region", "mes", "real_mwh", "prog_mwh", "desv_mwh", "desv_pct"]].head(15)
          .assign(desv_pct=lambda t: (100 * t.desv_pct).round(1)).round(0))
print(f"Mayores desviaciones DIARIAS: {d.supera_umbral.sum()} de {d.desv_pct.notna().sum()} planta-día pasan el {UMBRAL_DESV:.0%}")
display(d.sort_values("desv_pct", key=abs, ascending=False)
          [["nombre_xm", "region", "fecha", "real_mwh", "prog_mwh", "desv_mwh", "desv_pct"]].head(15)
          .assign(desv_pct=lambda t: (100 * t.desv_pct).round(1), fecha=lambda t: t.fecha.dt.date).round(1))

### 8.3 Mapas y gráficos de las desviaciones
Requiere haber ejecutado la sección 7 (usa sus funciones de mapa).

In [ ]:
from matplotlib.colors import TwoSlopeNorm
if d.empty:
    raise RuntimeError("No hay desviaciones diarias: ejecuta primero 8.1 y revisa su mensaje")
def guardar_desv(fig, nombre):
    fig.savefig(CARP_DESV / nombre, dpi=200, bbox_inches="tight", facecolor="white"); plt.show()

pp = plantas[["codigo_xm", "geometry"]].merge(por_planta.reset_index(), on="codigo_xm")
tam = 20 + 1.2 * pp["capacidad_mw"].fillna(pp["capacidad_mw"].median())

# Mapa D1 · tamaño típico de la desviación diaria en cada planta
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
mm = ax.scatter(pp.geometry.x, pp.geometry.y, s=tam, c=pp["desv_abs_media_pct_d"], cmap="Reds", vmin=0,
                edgecolor="k", linewidth=0.5, zorder=3)
barra(fig, ax, mm, "Desviación diaria media |real − programada| (%)"); guardar_desv(fig, "D1_desviacion_diaria_media.png")

# Mapa D2 · % de días en que la desviación pasa el 15 %
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
mm = ax.scatter(pp.geometry.x, pp.geometry.y, s=tam, c=pp["pct_sobre_umbral_d"], cmap="Purples", vmin=0, vmax=100,
                edgecolor="k", linewidth=0.5, zorder=3)
barra(fig, ax, mm, f"Días con |desviación| > {UMBRAL_DESV:.0%} (% de los días)"); guardar_desv(fig, "D2_dias_sobre_15pct.png")

# Mapa D3 · sesgo: ¿la planta produce más (+) o menos (−) de lo programado?
lim = max(5, float(np.nanmax(np.abs(pp["sesgo_pct_m"]))) if pp["sesgo_pct_m"].notna().any() else 5)
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
mm = ax.scatter(pp.geometry.x, pp.geometry.y, s=tam, c=pp["sesgo_pct_m"], cmap="RdBu",
                norm=TwoSlopeNorm(vcenter=0, vmin=-lim, vmax=lim), edgecolor="k", linewidth=0.5, zorder=3)
barra(fig, ax, mm, "Sesgo mensual medio (%) · rojo = produce menos de lo programado"); guardar_desv(fig, "D3_sesgo_mensual.png")

# Mapa D4 · departamentos: % de meses que pasan el 15 %
dd = dptos.merge(por_dpto["pct_sobre_umbral_m"].rename("pct_m").reset_index(), left_on="shapeName", right_on="departamento", how="left")
fig, ax = plt.subplots(figsize=FIG)
dd.plot(ax=ax, column="pct_m", cmap="OrRd", vmin=0, vmax=100, edgecolor="none",
        missing_kwds={"color": "#F2F2EF"})
base(ax); pp.plot(ax=ax, color="k", marker="^", markersize=14, zorder=3)
barra(fig, ax, plt.cm.ScalarMappable(cmap="OrRd", norm=plt.Normalize(0, 100)),
      f"Meses con |desviación| > {UMBRAL_DESV:.0%} por departamento (%)"); guardar_desv(fig, "D4_departamentos_meses_sobre_15pct.png")

# Gráfico · ranking de plantas (posiciones numéricas: los nombres repetidos no se mezclan)
r = por_planta.dropna(subset=["pct_sobre_umbral_d"]).sort_values("pct_sobre_umbral_d")
etiquetas = r["nombre_xm"].fillna(pd.Series(r.index, index=r.index)).astype(str).str.slice(0, 35)
y = np.arange(len(r))
fig, ax = plt.subplots(figsize=(7, 0.28 * len(r) + 1.5))
ax.barh(y, r["pct_sobre_umbral_d"], color="#B5543C", label="días")
ax.scatter(r["pct_sobre_umbral_m"], y, color="k", zorder=3, s=18, label="meses")
ax.set_yticks(y, etiquetas, fontsize=7)
ax.set_xlabel(f"% de periodos con |desviación| > {UMBRAL_DESV:.0%}"); ax.legend(loc="lower right")
ax.set_title("Plantas con más desviaciones grandes", loc="left", fontweight="bold")
guardar_desv(fig, "G1_ranking_plantas.png")

# Gráfico · ciclo diario: desviación media por hora del día y mes, por región
regs = [x for x in REGIONES if x in set(h["region"])]          # solo plantas con ubicación
fig, axs = plt.subplots(1, len(regs), figsize=(4.2 * len(regs), 3.6), squeeze=False)
for ax, reg in zip(axs[0], regs):
    t = (h[h.region == reg].assign(mes_año=lambda x: x.fecha_hora.dt.month)
           .pivot_table(index="hora", columns="mes_año", values="desv_pct", aggfunc="mean")
           .reindex(columns=range(1, 13)) * 100)
    im = ax.imshow(t.values, aspect="auto", cmap="RdBu", norm=TwoSlopeNorm(vcenter=0, vmin=-40, vmax=40),
                   extent=[0.5, 12.5, t.index.max() + 0.5, t.index.min() - 0.5])
    ax.set_title(reg, fontweight="bold"); ax.set_xlabel("Mes"); ax.set_ylabel("Hora del día"); ax.set_xticks(range(1, 13))
fig.colorbar(im, ax=axs[0].tolist(), label="Desviación media (%)", shrink=0.9)
guardar_desv(fig, "G2_ciclo_horario_por_region.png")

# Gráfico · evolución mensual por región (desviación ponderada por energía)
ev = m[m.region != "Sin ubicación"].groupby(["region", "mes"]).apply(lambda t: 100 * t.desv_mwh.sum() / t.prog_mwh.sum()).unstack(0)
fig, ax = plt.subplots(figsize=(8, 3.6))
ev.plot(ax=ax, marker="o", ms=3)
ax.axhspan(-100 * UMBRAL_DESV, 100 * UMBRAL_DESV, color="0.9", zorder=0, label="±15 %")
ax.axhline(0, color="k", lw=0.6); ax.set_ylabel("Desviación (%)"); ax.set_xlabel("")
ax.set_title("Desviación mensual real − programada por región", loc="left", fontweight="bold")
guardar_desv(fig, "G3_evolucion_mensual_regiones.png")
print("Mapas y gráficos en", CARP_DESV)
